In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

C:\Users\dheer\anaconda3\envs\spark311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkPractice") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

4.2.0


In [4]:
emp_data = [
(1,'manish',26,20000,'india','IT'),
(2,'rahul',None,40000,'germany','engineering'),
(3,'pawan',12,60000,'india','sales'),
(4,'roshini',44,None,'uk','engineering'),
(5,'raushan',35,70000,'india','sales'),
(6,None,29,200000,'uk','IT'),
(7,'adam',37,65000,'us','IT'),
(8,'chris',16,40000,'us','sales'),
(None,None,None,None,None,None),
(7,'adam',37,65000,'us','IT')
]

In [9]:
emp_df = spark.createDataFrame(emp_data, ['id','name','age','salary', 'county','dept'])

In [10]:
emp_df.show()

+----+-------+----+------+-------+-----------+
|  id|   name| age|salary| county|       dept|
+----+-------+----+------+-------+-----------+
|   1| manish|  26| 20000|  india|         IT|
|   2|  rahul|NULL| 40000|germany|engineering|
|   3|  pawan|  12| 60000|  india|      sales|
|   4|roshini|  44|  NULL|     uk|engineering|
|   5|raushan|  35| 70000|  india|      sales|
|   6|   NULL|  29|200000|     uk|         IT|
|   7|   adam|  37| 65000|     us|         IT|
|   8|  chris|  16| 40000|     us|      sales|
|NULL|   NULL|NULL|  NULL|   NULL|       NULL|
|   7|   adam|  37| 65000|     us|         IT|
+----+-------+----+------+-------+-----------+



In [11]:
emp_df.withColumn("adult", when(col('age')<18,"NO")         ## (coln name , when(col(schema)conditon))
                  .when(col('age')>18,"YES")
                   .otherwise("No value")).show()

+----+-------+----+------+-------+-----------+--------+
|  id|   name| age|salary| county|       dept|   adult|
+----+-------+----+------+-------+-----------+--------+
|   1| manish|  26| 20000|  india|         IT|     YES|
|   2|  rahul|NULL| 40000|germany|engineering|No value|
|   3|  pawan|  12| 60000|  india|      sales|      NO|
|   4|roshini|  44|  NULL|     uk|engineering|     YES|
|   5|raushan|  35| 70000|  india|      sales|     YES|
|   6|   NULL|  29|200000|     uk|         IT|     YES|
|   7|   adam|  37| 65000|     us|         IT|     YES|
|   8|  chris|  16| 40000|     us|      sales|      NO|
|NULL|   NULL|NULL|  NULL|   NULL|       NULL|No value|
|   7|   adam|  37| 65000|     us|         IT|     YES|
+----+-------+----+------+-------+-----------+--------+



In [15]:
from pyspark.sql.functions import col, when, lit

emp_df \
    .withColumn(
        "age",
        when(col("age").isNull(), lit(19))
        .otherwise(col("age"))
    ) \
    .withColumn(
        "adult",
        when(col("age") > 18, "YES")
        .otherwise("NO")
    ) \
    .show()

+----+-------+---+------+-------+-----------+-----+
|  id|   name|age|salary| county|       dept|adult|
+----+-------+---+------+-------+-----------+-----+
|   1| manish| 26| 20000|  india|         IT|  YES|
|   2|  rahul| 19| 40000|germany|engineering|  YES|
|   3|  pawan| 12| 60000|  india|      sales|   NO|
|   4|roshini| 44|  NULL|     uk|engineering|  YES|
|   5|raushan| 35| 70000|  india|      sales|  YES|
|   6|   NULL| 29|200000|     uk|         IT|  YES|
|   7|   adam| 37| 65000|     us|         IT|  YES|
|   8|  chris| 16| 40000|     us|      sales|   NO|
|NULL|   NULL| 19|  NULL|   NULL|       NULL|  YES|
|   7|   adam| 37| 65000|     us|         IT|  YES|
+----+-------+---+------+-------+-----------+-----+



In [20]:
emp_df.withColumn(
         "Status",      ## coln name
         when(col("age") < 18, "Minor") ##condition
         .otherwise("Major")  ## else
     ).show()

+----+-------+----+------+-------+-----------+------+
|  id|   name| age|salary| county|       dept|Status|
+----+-------+----+------+-------+-----------+------+
|   1| manish|  26| 20000|  india|         IT| Major|
|   2|  rahul|NULL| 40000|germany|engineering| Major|
|   3|  pawan|  12| 60000|  india|      sales| Minor|
|   4|roshini|  44|  NULL|     uk|engineering| Major|
|   5|raushan|  35| 70000|  india|      sales| Major|
|   6|   NULL|  29|200000|     uk|         IT| Major|
|   7|   adam|  37| 65000|     us|         IT| Major|
|   8|  chris|  16| 40000|     us|      sales| Minor|
|NULL|   NULL|NULL|  NULL|   NULL|       NULL| Major|
|   7|   adam|  37| 65000|     us|         IT| Major|
+----+-------+----+------+-------+-----------+------+



In [24]:
emp_df.createOrReplaceTempView("emp") ##register your DataFrame as a temporary view

In [23]:
spark.sql("""
    SELECT *,
           CASE
               WHEN age < 18 THEN 'Minor'
               ELSE 'Major'
           END AS Status
    FROM emp
""").show()

+----+-------+----+------+-------+-----------+------+
|  id|   name| age|salary| county|       dept|Status|
+----+-------+----+------+-------+-----------+------+
|   1| manish|  26| 20000|  india|         IT| Major|
|   2|  rahul|NULL| 40000|germany|engineering| Major|
|   3|  pawan|  12| 60000|  india|      sales| Minor|
|   4|roshini|  44|  NULL|     uk|engineering| Major|
|   5|raushan|  35| 70000|  india|      sales| Major|
|   6|   NULL|  29|200000|     uk|         IT| Major|
|   7|   adam|  37| 65000|     us|         IT| Major|
|   8|  chris|  16| 40000|     us|      sales| Minor|
|NULL|   NULL|NULL|  NULL|   NULL|       NULL| Major|
|   7|   adam|  37| 65000|     us|         IT| Major|
+----+-------+----+------+-------+-----------+------+

